In [1]:
import os
import pandas as pd 
import matplotlib.pyplot as plt 
import plotly
from pathlib import Path
import re
import unicodedata

In [2]:
carpeta_contenedora = '../datos_reales_cliente/'
archivo_ventas = 'Reportes de ventas 2026 semana 34 OK.xlsx'
archivo_plan = 'Plan presupuesto de venta LT Semanalizado 2026 clean.xlsx'
path_archivo_ventas = Path(carpeta_contenedora + archivo_ventas)
path_archivo_presupuesto = Path(carpeta_contenedora + archivo_plan)


In [3]:
year_ventas = re.search(r'\b(19|20)\d{2}\b', archivo_ventas).group(0)
year_presupuesto = re.search(r'\b(19|20)\d{2}\b', archivo_plan).group(0)
year_presupuesto

'2026'

In [4]:
## Definición para limpiar columnas 
def clean_df_columns_simple(df):
    old_cols = df.columns
    new_cols = [re.sub('_/_', '_', re.sub(' ', '_', column.lower())) for column in old_cols] 
    new_cols = [column.replace('(', '').replace(')', '') for column in new_cols]
    df.columns = new_cols
    return df 


In [5]:
def clean_df_columns(df):
    new_cols = []
    for column in df.columns:
        cleaned = column.lower()
        cleaned = unicodedata.normalize('NFKD', cleaned)
        cleaned = cleaned.encode('ascii', 'ignore').decode('utf-8')
        cleaned = re.sub(r'[\s/]+', '_', cleaned)  
        cleaned = re.sub(r'[()]', '', cleaned)              
        new_cols.append(cleaned)
        
    df.columns = new_cols
    return df


In [6]:
def delete_empty_cells_based_on_column(df, column):
    df = df[~df[column].isna()]
    return df 


In [7]:
df_ventas = pd.read_excel(path_archivo_ventas)
df_presupuesto = pd.read_excel(path_archivo_presupuesto)

In [8]:
df_ventas = clean_df_columns(df_ventas)
df_presupuesto = clean_df_columns(df_presupuesto)

In [9]:
df_ventas.columns

Index(['mes', 'folio_fiscal', 'tipo_comprobante', 'categoria', 'semana', 'rfc',
       'receptor', 'cajas_9_lts', 'unidades', 'sku', 'precio_unitario',
       'venta', 'ieps', 'iva', 'venta_+_ieps_+_iva', 'ceco_finanzas',
       'ceco_datalap', 'cc', 'serie', 'fecha_de_emision', 'monto', 'moneda',
       'estatus', 'punto_de_venta', 'canal', 'canal_reporte', 'cliente',
       'vendedor', 'ubicacion', 'costo_unitario', 'costo_total', 'margen'],
      dtype='object')

In [10]:
df_presupuesto.columns

Index(['#', 'region', 'canal', 'cliente', 'vendedor', 'mes', 'semana',
       'articulo', 'precio_bot', 'cogs', 'unidades', 'venta_neta_mxn',
       'cogs_total_mxn'],
      dtype='object')

In [11]:
df_ventas = delete_empty_cells_based_on_column(df_ventas, column='folio_fiscal')

In [12]:
df_ventas['year'] = year_ventas
df_presupuesto['year'] = year_presupuesto
df_ventas['uncleaned_date'] = df_ventas['year'].astype(str) + ' ' + df_ventas['mes'].astype(str) + ' ' + df_ventas['semana'].astype(str)
df_presupuesto['uncleaned_date'] = df_presupuesto['year'].astype(str) + ' ' + df_presupuesto['mes'].astype(str) + ' ' +  df_presupuesto['semana'].astype(str)

In [13]:
df_presupuesto.head(5)

,#,region,canal,cliente,vendedor,mes,semana,articulo,precio_bot,cogs,unidades,venta_neta_mxn,cogs_total_mxn,year,uncleaned_date
0,1,CDMX,OFF TRADE,PALACIO DE HIERRO,GALA ROSALES,Enero,1,LOCO TEQUILA BLANCO 200 ML,417.0,265.32,0.0,0.0,0.0,2026,2026 Enero 1
1,2,CDMX,OFF TRADE,PALACIO DE HIERRO,GALA ROSALES,Enero,1,LOCO TEQUILA BLANCO 750 ML,1098.7,442.31,0.0,0.0,0.0,2026,2026 Enero 1
2,3,CDMX,OFF TRADE,PALACIO DE HIERRO,GALA ROSALES,Enero,1,LOCO TEQUILA REPOSADO ÁMBAR 750 ML,1714.0,645.61,0.0,0.0,0.0,2026,2026 Enero 1
3,4,CDMX,OFF TRADE,PALACIO DE HIERRO,GALA ROSALES,Enero,1,LOCO TEQUILA PURO CORAZÓN 750 ML,2810.0,551.63,0.0,0.0,0.0,2026,2026 Enero 1
4,5,CDMX,OFF TRADE,PALACIO DE HIERRO,GALA ROSALES,Enero,1,LOCO TEQUILA AÑEJO ÁUREO 750 ML,6592.0,645.61,0.0,0.0,0.0,2026,2026 Enero 1


In [14]:
df_ventas.head(5)

,mes,folio_fiscal,tipo_comprobante,categoria,semana,rfc,receptor,cajas_9_lts,unidades,sku,...,canal,canal_reporte,cliente,vendedor,ubicacion,costo_unitario,costo_total,margen,year,uncleaned_date
0,1.0,2e04b129-96e4-46f9-a2c2-f79122e60c0b,FACTURA,Cancelado,Semana 01,GBS170519F86,GASTRONOMICA BURRO E SALVIA,0.000000,0,Loco Blanco,...,On Trade,On Trade,Parker And Lenox,Luis Franklin,CDMX,0.00,0.00,0.0000,2026,2026 1.0 Semana 01
1,1.0,f6beb10b-a1ff-488c-bdac-410e3543bf44,FACTURA,Producto (Botellas),Semana 01,GBS170519F86,GASTRONOMICA BURRO E SALVIA,0.166667,2,Loco Blanco,...,On Trade,On Trade,Sartoria,Luis Franklin,CDMX,518.20,1036.40,1780.8180,2026,2026 1.0 Semana 01
2,1.0,81d9268a-ad9d-4c7d-8510-771453435f95,FACTURA,Venta Activo,Semana 02,XAXX010101000,CHRISTIAN JASMIN HERNANDEZ ORDUÑO,0.000000,0,Venta de Tiguan,...,Administración,Administración,Christian Jasmin Hernandez,Administración,CDMX,0.00,0.00,0.0000,2026,2026 1.0 Semana 02
3,1.0,332f17b9-a1a2-4726-899a-96d0d18c375a,FACTURA,Producto (Botellas),Semana 02,GHO161123F23,GHSM HOSPITALITY,0.083333,1,Loco Blanco,...,On Trade,On Trade,GHSM Hunan Satelite,Rodrigo Luna,CDMX,518.20,518.20,721.3760,2026,2026 1.0 Semana 02
4,1.0,57c77fc6-7196-407e-9463-81ecc5c5c4f0,FACTURA,Producto (Botellas),Semana 02,GOM0603174R7,GOMCARLU,0.083333,1,Puro Corazon,...,On Trade,On Trade,Blanco Castelar,Rodrigo Luna,CDMX,817.86,817.86,2788.1801,2026,2026 1.0 Semana 02


In [15]:
df_ventas.to_csv('./datos_ventas_test.csv', index=False)
df_presupuesto.to_csv('./datos_presupuesto_test.csv', index=False)